<div style="border: none; margin: 5px 0; border-top: 1px dashed #FFFFFF; border-bottom: 1px dashed #FFFFFF; height: 5px;"></div>

<h2 style="color: #FFA07A;">6. Aplicação do modelo supervisionado</h2></h2>

In [2]:
#
#--- Import necessary libraries: ipywidgets for interactivity and time for delay simulation ---
import ipywidgets as widgets
import time

#--- Create an HTML widget to display the typing effect ---
output = widgets.HTML(value="<div></div>")
display(output)

#--- Define the formatted HTML text (in European Portuguese) ---
texto = """
<div>
    <p><b>Análise preditiva de serviços urbanos:</b></p> 
    <p>🔸 Nesta secção, vamos explorar o potencial dos modelos de <b>aprendizagem automática supervisionada</b> para melhorar a análise dos agrupamentos realizados pelo <b>K-Means</b>. Através destas técnicas, é possível não só identificar padrões, mas também prever e quantificar as relações entre variáveis que influenciam os serviços urbanos. A primeira etapa consiste em treinar três modelos preditivos com os dados dos <b><i>clusters</i></b>, proporcionando uma visão mais aprofundada do comportamento dos dados.</p> 
    <p>🔸 Ao aplicarmos modelos de <b>aprendizagem supervisionada</b>, que utilizam dados com rótulos conhecidos para realizar previsões, aumentamos a precisão dos resultados e compreendemos melhor as interacções entre variáveis. Esta abordagem permite-nos não só identificar tendências, como também antecipar comportamentos futuros e optimizar a alocação de recursos no território.</p> 
    <p>🔸 <u><b>O primeiro passo consiste em testar três modelos preditivos sobre os dados, avaliando o seu desempenho e ajustando-os para obter as melhores previsões.</b></u></p>
</div>
"""

#--- Prepare the base HTML structure for styling and typing effect ---
texto_html = """
<div style="background-color: #FFFFFF; color: #333333; padding: 15px; 
            border-left: 5px solid #FFA500; font-family: Arial, sans-serif; 
            text-align: justify; font-size: 16px; line-height: 1.6;">
"""

#--- Simulate typing effect by adding one word at a time with delay ---
for palavra in texto.split():
    texto_html += palavra + " "
    output.value = texto_html + "</div>"
    time.sleep(0.10)  # Adjust speed 

#--- Ensure the final HTML is correctly closed and rendered ---
output.value = texto_html + "</div>"

HTML(value='<div></div>')

In [3]:
from IPython.display import Javascript, display
# hide-me
display(Javascript('window.cellVisibilityManager.hideCells();'))

# Run libraries
ipython = get_ipython()
ipython.run_line_magic("run", "bibliotecas.ipynb")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
# hide-me
display(Javascript('window.cellVisibilityManager.hideCells();'))

#--- Load the data ---
input_pkl_path = "df_servicos_clusters.pkl"
with open(input_pkl_path, 'rb') as pkl_file:
    data = pickle.load(pkl_file)
df_servicos = data['df']

#--- Show initial processing message ---
print("\nA calcular dados, aguarde...", end='\r')

#--- Apply logarithmic transformations ---
df_servicos["distancia_media_servicos_log"] = np.log1p(df_servicos["distancia_media_servicos"])
df_servicos["pop_64_mais_log"] = np.log1p(df_servicos["pop_64_mais"])

#--- Ensure coordinate columns exist (reproject to obtain correct centroids) ---
if "coord_x" not in df_servicos.columns or "coord_y" not in df_servicos.columns:
    df_geo = gpd.GeoDataFrame(df_servicos, geometry='geometry', crs="EPSG:4326").to_crs(epsg=3763)
    df_servicos["coord_x"] = df_geo.geometry.centroid.x
    df_servicos["coord_y"] = df_geo.geometry.centroid.y

#--- Define predictor variables and target variable ---
variaveis = [
    "Centro Saude", "Farmacias", "Supermercados",
    "Parques e jardins", "Hospitais", "CTT",
    "distancia_media_servicos_log", "pop_64_mais_log", "coord_x", "coord_y"
]

X = df_servicos[variaveis].fillna(0)
y = df_servicos["numero_servicos_proximos"]

#--- Split into training and testing sets ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

#--- Define evaluation function for metrics ---
def calcular_metricas(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    rmsle = np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))
    medape = np.median(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1))) * 100
    cvrmse = rmse / np.mean(y_true) * 100
    return [r2, rmse, mae, rmsle, medape, cvrmse]

#--- Define models to test ---
models = {
    "Random Forest": RandomForestRegressor(n_estimators=50, random_state=42),
    "SVR": SVR(kernel='rbf'),
    "XGBoost": XGBRegressor(n_estimators=50, learning_rate=0.1, max_depth=5, random_state=42)
}

#--- Evaluate each model and store metrics ---
metrics_results = {}
for nome, modelo in models.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    metrics_results[nome] = calcular_metricas(y_test, y_pred)

#--- Organize results into a DataFrame ---
results_df = pd.DataFrame(metrics_results).T.reset_index()
results_df.columns = ["Modelo", "R²", "RMSE", "MAE", "RMSLE", "MedAPE (%)", "CVRMSE (%)"]

#--- Clear previous message ---
print(" " * 50, end='\r')
print("\nDados calculados com sucesso, exibindo tabela...\n")

#--- Display the results table using tabulate ---
print(tabulate(results_df, headers=[f"{col:^15}" for col in results_df.columns],
               tablefmt='fancy_grid', showindex=False, colalign=["center"] * len(results_df.columns)))

#--- Final information message ---
print("\033[92m[INFO] Após análise, pode continuar.\033[0m")

<IPython.core.display.Javascript object>


                                                  
Dados calculados com sucesso, exibindo tabela...

╒═══════════════════╤═══════════════════╤═══════════════════╤═══════════════════╤═══════════════════╤═══════════════════╤═══════════════════╕
│      Modelo       │        R²         │       RMSE        │        MAE        │       RMSLE       │    MedAPE (%)     │    CVRMSE (%)     │
╞═══════════════════╪═══════════════════╪═══════════════════╪═══════════════════╪═══════════════════╪═══════════════════╪═══════════════════╡
│   Random Forest   │     0.999828      │      0.24382      │     0.0540464     │     0.014927      │         0         │      1.01686      │
├───────────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────┼───────────────────┤
│        SVR        │    -0.0220677     │      18.7996      │      13.6941      │      0.82661      │      56.0507      │      78.4046      │
├───────────────────┼───────────────────┼─────

In [5]:
#
import ipywidgets as widgets
import time

#--- Create an HTML widget to display the typing effect ---
output = widgets.HTML(value="<div></div>")
display(output)

#--- Short explanatory text in Portuguese (Portugal) with formatting ---
texto = """
<div>
    <p style="font-size: 18px; font-weight: bold; color: #6A0DAD; text-align: justify;">Resumo Explicativo:</p> 
    <p>
        🔹 <b>Random Forest</b> e <b>XGBoost</b> obtiveram os melhores resultados (R² ≈ 0.99, erros baixos).<br>
        🔹 O <b>XGBoost</b> (Extreme Gradient Boosting) foi selecionado como modelo nesta investigação devido à sua reconhecida eficiência, escalabilidade e robustez na previsão de padrões complexos. <br>
        🔹 Utiliza <b>boosting</b> com Árvores de Decisão e regularização:
    </p>
    <ul>
        <li><b>L1 (Lasso):</b> Elimina variáveis irrelevantes.</li>
        <li><b>L2 (Ridge):</b> Estabiliza os coeficientes.</li>
    </ul>
    <p>
        🔹 Permite identificar os factores que mais influenciam as previsões.<br>
        🔹 <b>Explicação das métricas:</b>
    </p>
    <ul>
        <li><b>R²:</b> qualidade do ajuste (ideal → 1).</li>
        <li><b>RMSE / MAE:</b> precisão absoluta (mais baixo = melhor).</li>
        <li><b>RMSLE / MedAPE:</b> erro proporcional, robusto a valores extremos.</li>
        <li><b>Bias / CVRMSE:</b> viés e consistência do modelo.</li>
    </ul>
    <p>🔹 Próximo passo: optimizar o <b>XGBoost</b> para modelar a <b>atratividade</b>.</p>
</div>
"""

#--- Build the styled container for the typing effect ---
texto_html = """
<div style="background-color: #FFFFFF; color: #333333; padding: 15px; 
            border-left: 5px solid #6A0DAD; font-family: Arial, sans-serif; 
            text-align: justify; font-size: 16px; line-height: 1.6;">
"""

#--- Simulate typing effect word by word ---
for palavra in texto.split():
    texto_html += palavra + " "
    output.value = texto_html + "</div>"  
    time.sleep(0.10)  

#--- Finalize the HTML content and close the div ---
output.value = texto_html + "</div>"

HTML(value='<div></div>')

<div style="border: none; margin: 5px 0; border-top: 1px dashed #FFFFFF; border-bottom: 1px dashed #FFFFFF; height: 5px;"></div>

In [6]:
#--- Import display tools from IPython to render HTML ---
from IPython.display import display, HTML

#--- HTML content for the section title with styling ---
html_content = '<h2 style="color: #FFA07A;">6.1. Cálculo de pesos da variável "atratividade" com análise de componentes principais (PCA)</h2>'

#--- Display the styled HTML heading ---
display(HTML(html_content))

In [38]:
#
#--- Import required libraries: ipywidgets for interactive HTML display, time for delay effect ---
import ipywidgets as widgets
import time

#--- Create an HTML widget to simulate typing effect ---
output = widgets.HTML(value="<div></div>")
display(output)

#--- Formatted explanatory text in Portuguese (Portugal) ---
texto = """
<div>
    <p><b>Análise de Componentes Principais (PCA):</b></p> 
    <p> 🔸 PCA é uma técnica estatística de transformação linear utilizada para reduzir a dimensão de um conjunto de variáveis correlacionadas, projectando‑as num novo espaço de variáveis não correlacionadas (os componentes principais), ordenadas pela quantidade de variância que cada uma explica.</p> 
    <p> 🔸 Neste estudo de caso, aplicou-se a PCA com <code>n_components=1</code> para obter um único vector de cargas (os coeficientes do primeiro componente). Estes coeficientes, após normalização, tornam-se os pesos utilizados na construção do índice de atratividade, uma vez que reflectem a importância relativa de cada variável na explicação da variância conjunta dos dados.</p>
</div>
"""

#--- Build the HTML container with styling for the typing effect ---
texto_html = """
<div style="background-color: #FFFFFF; color: #333333; padding: 15px; 
            border-left: 5px solid #FFA500; font-family: Arial, sans-serif; 
            text-align: justify; font-size: 16px; line-height: 1.6;">
"""

#--- Simulate typing by appending each word with a delay ---
for palavra in texto.split():
    texto_html += palavra + " "
    output.value = texto_html + "</div>"  
    time.sleep(0.10)   # Adjust speed 

#--- Finalize the HTML block ---
output.value = texto_html + "</div>"

HTML(value='<div></div>')

In [39]:
# hide-me
display(Javascript('window.cellVisibilityManager.hideCells();'))

# --- Winsorization function (in case you want to use it later) ---
def winsorize_series(s, lower_quantile=0.00, upper_quantile=0.95):
    lower = s.quantile(lower_quantile)
    upper = s.quantile(upper_quantile)
    return s.clip(lower, upper)

# --- List your variables of interest ---
vars_raw = [
    "Bancos", "Centro Saude", "Farmacias", "Supermercados",
    "Parques e jardins", "Hospitais", "CTT",
    "pop_64_mais", "distancia_media_servicos",
    "coord_x", "coord_y"
]

# --- Create a DataFrame with only the selected variables and handle NAs ---
df_idx = df_servicos[vars_raw].fillna(0).copy()

# Create the inverse distance column and drop the original one
df_idx["invdist"] = 1 / (df_idx["distancia_media_servicos"] + 1)
df_idx.drop("distancia_media_servicos", axis=1, inplace=True)

# Normalize each variable to the [0, 1] range
scaler = MinMaxScaler()
X_norm = scaler.fit_transform(df_idx)  # shape = (n_samples, 9)

# --- Obtain a single component via PCA and use its loadings as weights ---
pca = PCA(n_components=1, random_state=42)
pca.fit(X_norm)
pesos_raw = np.abs(pca.components_[0])
pesos = pesos_raw / pesos_raw.sum()

# To quickly check the result
nomes = df_idx.columns.tolist()
pesos_dict = dict(zip(nomes, np.round(pesos, 4)))

# --- Compute the attractiveness index as a weighted sum ---
df_servicos["atratividade"] = (X_norm * pesos).sum(axis=1)

# Log transform + Winsorization to tame outliers
df_servicos["atratividade"] = np.log1p(df_servicos["atratividade"])
df_servicos["atratividade"] = winsorize_series(df_servicos["atratividade"], 0.00, 0.95)

# --- Friendly labels for display ---
label_mapping = {
    'Bancos': 'Bancos',
    'Centro Saude': 'Centro de saúde',
    'Farmacias': 'Farmácias',
    'Supermercados': 'Supermercados',
    'Parques e jardins': 'Parques ou jardins',
    'Hospitais': 'Hospitais',
    'CTT': 'CTT',
    'pop_64_mais': 'População com 65 anos ou +',
    'coord_x': 'Coordenada X',
    'coord_y': 'Coordenada Y',
    'invdist': 'Inverso da distância média'
}

# --- Display normalized weights in a sorted table ---
df_pesos = pd.DataFrame({
    "Variável": [label_mapping.get(v, v) for v in nomes],
    "Peso": np.round(pesos, 4)
}).sort_values("Peso", ascending=False).reset_index(drop=True)

print("Pesos normalizados das variáveis (ordenados):")
display(df_pesos)

# --- Display the first 10 values of the attractiveness index ---
tabela_resultados = df_servicos[["atratividade"]].copy().round(4)

<IPython.core.display.Javascript object>

Pesos normalizados das variáveis (ordenados):


,Variável,Peso
0,Bancos,0.1663
1,Supermercados,0.1512
2,CTT,0.1459
3,Centro de saúde,0.1420
4,Farmácias,0.1373
5,Parques ou jardins,0.1296
6,Coordenada Y,0.0651
7,Hospitais,0.0473
8,Coordenada X,0.0122
9,Inverso da distância média,0.0018


<div style="border: none; margin: 5px 0; border-top: 1px dashed #FFFFFF; border-bottom: 1px dashed #FFFFFF; height: 5px;"></div>

In [9]:
#--- Import display utilities from IPython to render HTML ---
from IPython.display import display, HTML

#--- HTML content for section title with custom color styling ---
html_content = '<h2 style="color: #FFA07A;">6.2. Cálculo de pesos da variável "atratividade" com análise fatorial confirmatória (CFA)</h2>'

#--- Display the styled HTML heading ---
display(HTML(html_content))

In [10]:
#
#--- Import necessary libraries: ipywidgets for output, time for typing effect ---
import ipywidgets as widgets
import time

#--- Create an HTML widget to simulate the typing effect ---
output = widgets.HTML(value="<div></div>")
display(output)

#--- Well-formatted explanatory text in Portuguese (Portugal) ---
texto = """
<div>
    <p><b>Análise Fatorial Confirmatória (CFA):</b></p> 
    <p> 🔸 A CFA ajusta um modelo de equações estruturais em que uma variável latente (“Atratividade”) é medida pelos nove indicadores de serviços utilizados na análise PCA.</p> 
    <p><b> Para que serve:</b></p>
    <p> 🔸 As cargas fatoriais mostram a relevância de cada serviço na definição do fator “Atratividade” (ex.: se “farmácias” tem uma carga padronizada de 0.8, é um bom indicador do fator).</p>
</div>
"""

#--- Create styled container for typing effect while preserving HTML formatting ---
texto_html = """
<div style="background-color: #FFFFFF; color: #333333; padding: 15px; 
            border-left: 5px solid #FFA500; font-family: Arial, sans-serif; 
            text-align: justify; font-size: 16px; line-height: 1.6;">
"""

#--- Simulate typing effect by appending each word with delay ---
for palavra in texto.split():
    texto_html += palavra + " "
    output.value = texto_html + "</div>"  
    time.sleep(0.10)  

#--- Finalize and close the HTML structure ---
output.value = texto_html + "</div>"

HTML(value='<div></div>')

In [43]:
# hide-me
display(Javascript('window.cellVisibilityManager.hideCells();'))

# --- Variables and data preparation ---
vars_raw = [
    "Bancos", "Centro Saude", "Farmacias", "Supermercados",
    "Parques e jardins", "Hospitais", "CTT", "pop_64_mais", "distancia_media_servicos",
    "coord_x", "coord_y"
]

# Create DataFrame with missing values filled
df_idx = df_servicos[vars_raw].fillna(0).copy()

# Create 'invdist' variable as the inverse of the average distance to services
df_idx["invdist"] = 1 / (df_idx["distancia_media_servicos"] + 1)
df_idx.drop("distancia_media_servicos", axis=1, inplace=True)

# Normalize the variables to the [0, 1] range
scaler = MinMaxScaler()
X_norm = scaler.fit_transform(df_idx)

# Apply cleaned names to columns (replacing `+` and spaces with `_`)
colunas_corrigidas = [
    'Bancos', 'Centro_de_saude', 'Farmacias', 'Supermercados',
    'Parques_ou_jardins', 'Hospitais', 'CTT', 'Populacao_com_65_anos_ou_mais', 
    'Coordenada_X', 'Coordenada_Y', 'Inverso_da_distancia_media'
]

# Create DataFrame with renamed columns
df_afc = pd.DataFrame(X_norm, columns=colunas_corrigidas)

# --- Define CFA model including coordinates ---
modelo_afc = (
    "Atratividade =~ Bancos + Centro_de_saude + Farmacias + Supermercados + "
    "Parques_ou_jardins + Hospitais + CTT + Populacao_com_65_anos_ou_mais + "
    "Coordenada_X + Coordenada_Y + Inverso_da_distancia_media"
)

# --- Fit the model ---
mod = Model(modelo_afc)
mod.load_dataset(df_afc)  # Column names must match those used in the model
mod.fit()

# --- Formatted results ---
cargas_fatoriais = mod.inspect(std_est=True)

# Rename columns for clearer display
cargas_formatadas = cargas_fatoriais[cargas_fatoriais["op"] == "~"][
    ["lval", "rval", "Est. Std"]
]

# Renomear as colunas para exibir de forma legível
cargas_formatadas.columns = ["Latente", "Variável", "Carga Fatorial Padronizada"]

# Sort by standardized factor loadings in descending order (similar to PCA)
cargas_formatadas = cargas_formatadas.sort_values(by="Carga Fatorial Padronizada", ascending=False).reset_index(drop=True)

# Display formatted factor loadings
display(cargas_formatadas)

# Compute model fit indices
indices_ajuste = calc_stats(mod).T.reset_index()
indices_ajuste.columns = ["Métrica", "Valor"]

# Display model fit indices
display(indices_ajuste)

<IPython.core.display.Javascript object>

,Latente,Variável,Carga Fatorial Padronizada
0,Supermercados,Atratividade,0.998700
1,Bancos,Atratividade,0.926545
2,CTT,Atratividade,0.885386
3,Farmacias,Atratividade,0.875099
4,Centro_de_saude,Atratividade,0.769755
5,Parques_ou_jardins,Atratividade,0.724632
6,Hospitais,Atratividade,0.393541
7,Coordenada_X,Atratividade,0.140478
8,Inverso_da_distancia_media,Atratividade,-0.031304
9,Populacao_com_65_anos_ou_mais,Atratividade,-0.114876


,Métrica,Valor
0,DoF,44.000000
1,DoF Baseline,55.000000
2,chi2,35305.835664
3,chi2 p-value,0.000000
4,chi2 Baseline,260045.436000
5,CFI,0.864373
6,GFI,0.864232
7,AGFI,0.830290
8,NFI,0.864232
9,TLI,0.830466


In [8]:
#--- Import libraries for widget display and time delay ---
import ipywidgets as widgets
import time

#--- Create an HTML widget to simulate the typing effect ---
output = widgets.HTML(value="<div></div>")
display(output)

#--- Summary text with explanatory content
texto = """
<div>
    <p style="font-size: 18px; font-weight: bold; color: #6A0DAD; text-align: justify;">Resumo dos resultados:</p> 
    <p>🔹 Os pesos obtidos pela PCA e as cargas fatoriais da CFA estão alinhados, o que confirma a coerência da estrutura latente do fator "Atratividade".</p>
    <p>🔹 Variáveis como <b>supermercados</b>, <b>CTT</b>, <b>bancos</b> e <b>farmácias</b> apresentam cargas fatoriais elevadas (&gt; 0.85), sendo assim os melhores indicadores do fator.</p>
    <p>🔹 Já <b>hospitais</b>, <b>população com 65 anos ou +</b>, <b>inverso da distância média aos serviços</b> e <b>coordenada Y</b> registam cargas mais baixas (ou até negativas), indicando menor ou inversa contribuição para a atratividade.</p>
</div>
"""

#--- Create styled HTML container for typing effect ---
texto_html = """
<div style="background-color: #FFFFFF; color: #333333; padding: 15px; 
            border-left: 5px solid #6A0DAD; font-family: Arial, sans-serif; 
            text-align: justify; font-size: 16px; line-height: 1.6;">
"""

#--- Simulate typing effect by adding one word at a time with delay ---
for palavra in texto.split():
    texto_html += palavra + " "
    output.value = texto_html + "</div>"
    time.sleep(0.10) # Adjust speed 

#--- Finalize and display the full text ---
output.value = texto_html + "</div>"

HTML(value='<div></div>')

<div style="border: none; margin: 5px 0; border-top: 1px dashed #FFFFFF; border-bottom: 1px dashed #FFFFFF; height: 5px;"></div>

Seguinte: [Aplicação do XGBoost](7.ipynb)

<div style="border: none; margin: 5px 0; border-top: 1px dashed #FFFFFF; border-bottom: 1px dashed #FFFFFF; height: 5px;"></div>